In [10]:
# importações
import pandas as pd
import numpy as np

In [11]:
# Carregando base de dados
df = pd.read_csv('../data/raw/base_rh_ptBR.csv')

In [12]:
# Ajustando colunas temporais
COLS_NUMERICAS = [
    'Idade',
    'Valor_Diaria',
    'Distancia_de_Casa',
    'Valor_Hora',
    'Renda_Mensal',
    'Taxa_Mensal',
    'Num_Empresas_Trabalhadas',
    'Percentual_Aumento_Salario',
    'Total_Anos_Trabalhados',
    'Treinamentos_Ano_Passado',
    'Anos_na_Empresa',
    'Anos_na_Funcao_Atual',
    'Anos_Desde_Ultima_Promocao',
    'Anos_com_Gerente_Atual'
]
# Escalas das colunas ordinais (do menor para o maior nível)

#ESCALA_QUALIDADE = ['Ruim', 'Regular', 'Médio', 'Bom', 'Excelente']
#ESCALA_ACABAMENTO_PORAO = [
#    'Sem acabamento', 'Baixa qualidade', 'Sala de recreação',
#    'Habitável abaixo da média', 'Habitável média', 'Habitável boa',
#]

ESCALAS_ORDINAIS = {
    'Nivel_de_Educacao':{  
        1: "Ensino fundamental",
        2: "Ensino medio",
        3: "Graduacao",
        4: "Mestrado",
        5: "Doutorado",
    }, 
    'Satisfacao_com_Ambiente':{  
        1: "Baixa",
        2: "Media",
        3: "Alta",
        4: "Muito Alta",
    },
    "Envolvimento_com_Trabalho": {
        1: "Baixo",
        2: "Medio",
        3: "Alto",
        4: "Muito Alto",
    },
    "Nivel_do_Cargo": {
        1: "Nível inicial",
        2: "Nível básico",
        3: "Nível intermediário",
        4: "Nível avançado",
        5: "Nível mais alto",
    },
    'Satisfacao_no_Trabalho':{  
        1: "Baixa",
        2: "Media",
        3: "Alta",
        4: "Muito Alta",
    },
    "Avaliacao_de_Desempenho":{
        1: "Baixo",
        2: "Bom",
        3: "Excelente",
        4: "Excepcional",
    },
    'Satisfacao_em_Relacionamentos':{  
        1: "Baixa",
        2: "Media",
        3: "Alta",
        4: "Muito Alta",
    },
    "Equilibrio_Vida_Pessoal_Profissional": {
        1: "Ruim",
        2: "Bom",
        3: "Melhor",
        4: "Otimo",
    },
    "Nivel_de_Opcoes_de_Acoes":{
        0: "Nenhuma opção de ações",
        1: "Baixa quantidade de opções",
        2: "Quantidade intermediária de opções",
        3: "Alta quantidade de opções",
    }
}

COLS_ORDINAIS = list(ESCALAS_ORDINAIS)

# lista de dicionários para armazenar informações sobre as colunas
col_info = []

# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]

    # tipo e valores (menor/maior nível, ou primeiro/último para nominais)
    if col in COLS_NUMERICAS or pd.api.types.is_datetime64_any_dtype(serie):
        col_type = 'Temporal'
        values = [serie.min(), serie.max()]
    elif col in COLS_ORDINAIS:
        col_type = 'Ordinal'
        escala = ESCALAS_ORDINAIS[col]
        if escala is None:
            values = [serie.min(), serie.max()]
        else:
            presentes = [nivel for nivel in escala if nivel in set(serie.dropna())]
            values = [presentes[0], presentes[-1]] if presentes else [None, None]
    elif serie.dtype == 'int64':
        col_type = 'Discreta'
        values = [serie.min(), serie.max()]
    elif serie.dtype == 'float64':
        col_type = 'Contínua'
        values = [serie.min(), serie.max()]
    else:  # object, str, category, bool
        col_type = 'Nominal'
        uniques = serie.dropna().unique()
        values = [uniques[0], uniques[-1]] if len(uniques) else [None, None]

    # descrição (vamos supor que você já tenha uma descrição)
    description = 'Descrição da coluna'

    # adicionando informações à lista de dicionários
    col_info.append({
        'variavel': col,
        'descricao': description,
        'valores': values,
        'Tipo': col_type
    })

# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

In [ ]:
# Descrições de negócio, por nome da coluna em português
DESCRICOES_RH = {
    "Idade": "Idade do funcionário em anos",
    "Desligamento": "Indica se o funcionário desligou-se da empresa (Sim/Não) - Variável Alvo",
    "Viagens_de_Negocios": "Frequência com que o funcionário viaja a trabalho",
    "Valor_Diaria": "Custo diário do funcionário / Valor de diária",
    "Departamento": "Departamento onde o funcionário atua (Vendas, P&D, RH)",
    "Distancia_de_Casa": "Distância entre a residência do funcionário e o local de trabalho",
    "Nivel_de_Educacao": "Nível de escolaridade (1: Abaixo da Faculdade, 2: Faculdade, 3: Bacharelado, 4: Mestrado, 5: Doutorado)",
    "Area_de_Formacao": "Área de formação acadêmica do funcionário",
    "Contagem_de_Funcionarios": "Contagem de funcionários (valor constante 1 para todos)",
    "Numero_do_Funcionario": "Identificador único (ID) do funcionário",
    "Satisfacao_com_Ambiente": "Satisfação com o ambiente de trabalho (1: Baixa, 2: Média, 3: Alta, 4: Muito Alta)",
    "Genero": "Gênero do funcionário",
    "Valor_Hora": "Valor da taxa por hora de trabalho",
    "Envolvimento_com_Trabalho": "Nível de envolvimento/engajamento com o trabalho (1: Baixo, 2: Médio, 3: Alto, 4: Muito Alto)",
    "Nivel_do_Cargo": "Nível hierárquico do cargo na empresa (1 a 5)",
    "Cargo": "Nome da função ou cargo que o funcionário ocupa",
    "Satisfacao_no_Trabalho": "Satisfação geral do funcionário com o trabalho (1: Baixa, 2: Média, 3: Alta, 4: Muito Alta)",
    "Estado_Civil": "Estado civil do funcionário",
    "Renda_Mensal": "Salário mensal bruto do funcionário",
    "Taxa_Mensal": "Taxa mensal / Variação salarial",
    "Num_Empresas_Trabalhadas": "Número de empresas em que o funcionário trabalhou anteriormente",
    "Maior_de_18": "Indica se o funcionário é maior de 18 anos (S/N)",
    "Hora_Extra": "Indica se o funcionário costuma fazer horas extras (Sim/Não)",
    "Percentual_Aumento_Salario": "Percentual de aumento salarial recebido no último ano",
    "Avaliacao_de_Desempenho": "Nota da avaliação de desempenho (3: Excelente, 4: Excepcional)",
    "Satisfacao_em_Relacionamentos": "Satisfação com os relacionamentos interpessoais no trabalho (1: Baixa, 2: Média, 3: Alta, 4: Muito Alta)",
    "Horas_Padrao": "Horas padrão de trabalho no período (geralmente constante 80)",
    "Nivel_de_Opcoes_de_Acoes": "Nível de opções de ações concedidas ao funcionário (0 a 3)",
    "Total_Anos_Trabalhados": "Total de anos de experiência profissional na carreira",
    "Treinamentos_Ano_Passado": "Número de vezes que o funcionário participou de treinamentos no ano passado",
    "Equilibrio_Vida_Pessoal_Profissional": "Percepção do equilíbrio entre vida pessoal e profissional (1: Ruim, 2: Bom, 3: Muito Bom, 4: Excelente)",
    "Anos_na_Empresa": "Tempo de serviço do funcionário na empresa atual (em anos)",
    "Anos_na_Funcao_Atual": "Tempo que o funcionário está no mesmo cargo/função atual (em anos)",
    "Anos_Desde_Ultima_Promocao": "Tempo decorrido desde a última promoção do funcionário (em anos)",
    "Anos_com_Gerente_Atual": "Tempo que o funcionário trabalha sob a gestão do mesmo líder/gerente atual (em anos)"
}

# Adicionando descrição para cada uma das variáveis selecionadas
descricoes = [DESCRICOES_RH[original] for original in col_info_df["variavel"]]

# Verificação para garantir que temos o mesmo número de descrições e linhas
assert len(descricoes) == len(col_info_df)

# Atribuindo a lista diretamente como uma nova coluna (uma forma um pouco mais pythonica/pandas de fazer do que o loop do professor)
col_info_df["descricao"] = descricoes

# Visualizando o resultado final
display(col_info_df)

KeyError: 'Valor_Diaria'

In [ ]:
# Salvando dicionário de dados
col_info_df["valores"] = col_info_df["valores"].apply(
    lambda valores: [valor.item() if isinstance(valor, np.generic) else valor for valor in valores]
)
col_info_df.to_csv('../data/processed/dicionario.csv',index=False,sep=';')

In [ ]:
# Salvando escalas ordinais
# Escala usada para as ordinais numéricas (marcadas como None no dicionário)
ESCALA_NUMERICA_PADRAO = list(range(1, 11))

# Convertendo o dicionário de escalas para o formato longo
linhas_escala = []
for coluna, niveis in ESCALAS_ORDINAIS.items():
    if niveis is None:
        niveis = ESCALA_NUMERICA_PADRAO
    for ordem, nivel in enumerate(niveis, start=1):
        linhas_escala.append({'coluna': coluna, 'nivel': nivel, 'ordem': ordem})

escala_df = pd.DataFrame(linhas_escala)

# Salvando o arquivo
escala_df.to_csv('../data/processed/escala.csv', index=False, encoding='utf-8', sep=';')